# 01 — Factor Momentum Foundations

> Esta investigación pregunta si parte del momentum observado en activos puede entenderse como exposición a factores o grupos sistemáticos que también presentan persistencia temporal.

$ R_{i,t} = \sum_{k=1}^{K} \beta_{ik}F_{k,t} + \epsilon_{i,t} $

La primera etapa usará proxies ETF observables y líquidos, no factores académicos puros. La ecuación es un marco de investigación; no supone todavía que los proxies expliquen los retornos de los activos.


## <b>Relación con TSMOM</b>

`01_tsmom` ya establece un baseline de momentum temporal. Allí la pregunta es: ¿el activo viene subiendo o bajando respecto a su propia historia? Aquí la pregunta cambia a: ¿ese momentum coincide con momentum persistente en factores o grupos sistemáticos?

Esta línea complementa el baseline; no lo reemplaza ni reutiliza todavía abstracciones de código comunes.


## <b>Datos</b>

> Todo análisis posterior leerá exclusivamente `data/gold/daily_series/etf/<SYMBOL>.parquet`. Raw, Silver, `series/` e intradía quedan fuera del análisis.

El universo proxy inicial propuesto es SPY, QQQ, IWM, IWD, IWF, XLE y XLF. La disponibilidad se comprobará antes de calcular retornos.


## <b>Block 01 — Factor universe</b>

> Verificar el universo de proxies ETF disponible en Gold, su cobertura común y los límites de interpretación de cada proxy.

Estos ETFs son proxies económicos observables. En esta primera etapa permiten estudiar persistencia relativa entre estilos y sectores sin asumir todavía una estructura factorial académica exacta.


In [ ]:
from datetime import date
from pathlib import Path

import plotly.graph_objects as go
import polars as pl

PROXIES = {
    "SPY": "broad US equity market",
    "QQQ": "large-cap growth / technology tilt",
    "IWM": "small-cap tilt",
    "IWD": "value tilt",
    "IWF": "growth tilt",
    "XLE": "energy sector",
    "XLF": "financial sector",
}
RESEARCH_START = date(2018, 1, 2)
RESEARCH_END = date(2026, 11, 17)

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        for project_root in (candidate, candidate / "Momentum_systematic_strategies"):
            if (project_root / "data" / "gold" / "daily_series" / "etf").is_dir():
                return project_root
    raise FileNotFoundError("No se encontró data/gold/daily_series/etf desde el directorio actual.")

PROJECT_ROOT = find_project_root(Path.cwd())
GOLD_ROOT = PROJECT_ROOT / "data" / "gold" / "daily_series" / "etf"


In [ ]:
frames = {}
audit_rows = []

for symbol, proxy_role in PROXIES.items():
    path = GOLD_ROOT / f"{symbol}.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Gold no disponible para {symbol}: {path}")

    frame = pl.read_parquet(path).with_columns(pl.col("session_date").cast(pl.Date))
    if "adjusted_close" not in frame.columns:
        raise ValueError(f"{symbol} no contiene adjusted_close.")

    frame = frame.filter(
        pl.col("session_date").is_between(RESEARCH_START, RESEARCH_END, closed="both")
    ).sort("session_date")
    dates = frame.get_column("session_date").to_list()
    relevant_nulls = int(frame.select(["session_date", "adjusted_close"]).null_count().sum_horizontal()[0])
    duplicates = int(frame.get_column("session_date").is_duplicated().sum())

    if not frame.height or relevant_nulls or duplicates or dates != sorted(dates):
        raise ValueError(f"{symbol} no supera los controles básicos de cobertura.")
    if bool((frame.get_column("adjusted_close") <= 0).any()):
        raise ValueError(f"{symbol} contiene adjusted_close no positivo.")

    frames[symbol] = frame.select("session_date", "adjusted_close")
    audit_rows.append({
        "symbol": symbol,
        "proxy_role": proxy_role,
        "start_date": frame.get_column("session_date").min(),
        "end_date": frame.get_column("session_date").max(),
        "n_sessions": frame.height,
        "nulls": relevant_nulls,
        "duplicates": duplicates,
    })

universe_audit = pl.DataFrame(audit_rows)
universe_audit


La tabla confirma `adjusted_close`, precios positivos, fechas ordenadas, ausencia de duplicados y nulos relevantes. El control de calidad detallado ya corresponde a la promoción Gold.


In [ ]:
common_panel = frames["SPY"].rename({"session_date": "date", "adjusted_close": "SPY"})
for symbol in list(PROXIES)[1:]:
    common_panel = common_panel.join(
        frames[symbol].rename({"session_date": "date", "adjusted_close": symbol}),
        on="date",
        how="inner",
    )

common_panel = common_panel.sort("date")
panel_summary = {
    "dimensions": common_panel.shape,
    "first_date": common_panel.get_column("date").min(),
    "last_date": common_panel.get_column("date").max(),
    "nulls": int(common_panel.null_count().sum_horizontal()[0]),
    "common_sessions": common_panel.height,
}
panel_summary


In [ ]:
normalized_prices = common_panel.select(
    "date",
    *[(pl.col(symbol) / pl.col(symbol).first()).alias(symbol) for symbol in PROXIES],
)

figure = go.Figure()
for symbol in PROXIES:
    figure.add_scatter(x=normalized_prices["date"], y=normalized_prices[symbol], mode="lines", name=symbol)

figure.update_layout(
    title="Factor proxy universe — normalized adjusted prices",
    yaxis_title="P_t / P_0",
    legend_title_text="",
)
figure.show()


### <b>Block 01 conclusion</b>

- Los siete proxies propuestos — SPY, QQQ, IWM, IWD, IWF, XLE y XLF — están disponibles en Gold.
- La ventana común exacta es `2018-01-02 → 2026-11-17`, con 2,232 sesiones.
- Cada serie conserva 2,232 observaciones, `adjusted_close` disponible, precios positivos y cero nulos o fechas duplicadas en la ventana.
- No quedan diferencias de cobertura entre los proxies dentro del rango de investigación.
- El panel `date | SPY | QQQ | IWM | IWD | IWF | XLE | XLF` queda preparado para construir factor returns.

El siguiente bloque estudiará los retornos diarios de estos proxies.


## <b>Block 02 — Factor returns</b>

> Construir retornos diarios causales a partir de los precios ajustados Gold de los proxies disponibles.

_Placeholder — sin cálculos todavía._


## <b>Block 03 — Factor momentum</b>

> Describir la persistencia temporal de los retornos de los proxies sin construir aún señales ni estrategias.

_Placeholder — sin cálculos todavía._


## <b>Block 04 — Asset vs factor momentum</b>

> Comparar de forma descriptiva el momentum de activos con el de los proxies disponibles, sin estimar todavía betas ni regresiones factoriales.

_Placeholder — sin cálculos todavía._


## <b>Block 05 — Diagnostics</b>

> Revisar cobertura, concentración, correlaciones y límites de los proxies antes de profundizar en la relación entre activos y factores.

_Placeholder — sin cálculos todavía._
